# معمل إعداد بيئة العمل والتحقق من مكتبات LangGraph
### Interactive Hands-on Lab: Environment & Workspace Readiness (Lecture 69)

أهلاً بك يا بطل في هذا المعمل العملي!
في هذا الدرس، سنقوم بتهيئة وفحص بيئة العمل بالكامل للتأكد من أن كود **LangGraph** سيعمل معك بسلاسة داخل VS Code دون أي مشاكل.

سنتعلم ونجرب عملياً:
1. فحص مفسر بايثون والتأكد من العمل داخل بيئة معزولة (Virtual Environment).
2. التحقق من تثبيت مكتبات LangGraph الأساسية وحزمة `ipykernel`.
3. فحص ملف المتغيرات السرية (`.env`) دون كشف المفاتيح في الشاشة.
4. فهم طريقة تفعيل التتبع ومراقبة الأداء عبر LangSmith.
5. تشغيل جراف فحص أوتوماتيكي بسيط للتأكد من جاهزية كل شيء.
6. قسم خاص: جرب بنفسك!


In [1]:
import sys
import os

print("مسار مفسر بايثون الحالي:")
print(sys.executable)
print("\nإصدار بايثون:")
print(sys.version.split()[0])


مسار مفسر بايثون الحالي:
D:\Code\AI\rag-mastery\.venv\Scripts\python.exe

إصدار بايثون:
3.12.12


### 1. فحص تثبيت المكتبات الأساسية
تعال نتأكد من أن المكتبات التي ذكرها المحاضر (langchain, langgraph, ipykernel) موجودة وجاهزة للعمل.


In [2]:
try:
    import langgraph
    import langchain_core
    import ipykernel
    print("كل المكتبات الأساسية مثبتة وجاهزة للعمل بنجاح!")
except ImportError as error:
    print("تنبيه: هناك مكتبة ناقصة:", error)


كل المكتبات الأساسية مثبتة وجاهزة للعمل بنجاح!


### 2. فحص ملف المتغيرات السرية (.env)
ملف `.env` هو المكان الآمن الذي نضع فيه مفاتيحنا حتى لا تتسرب في الكود.
تعال نفحص هل المفاتيح موجودة أم لا، بدون طباعة نص المفتاح نفسه لأسباب أمنية.


In [3]:
import dotenv

dotenv.load_dotenv()

groq_key = os.getenv("GROQ_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

print("حالة المفاتيح السرية في ملف .env:")
print("- مفتاح Groq:", "متوفر وجاهز" if groq_key else "غير موجود")
print("- مفتاح OpenAI:", "متوفر وجاهز" if openai_key else "غير موجود")


حالة المفاتيح السرية في ملف .env:
- مفتاح Groq: متوفر وجاهز
- مفتاح OpenAI: غير موجود


### 3. ما هو LangSmith ولماذا نحتاجه؟
في تطبيقات الوكلاء متعددة الخطوات، قد يدور الوكيل في حلقات تكرارية ويستدعي أدوات كثيرة.
خدمة LangSmith تسمح لنا بمشاهدة كل خطوة والتكلفة والوقت المستغرق بالملي ثانية.


In [4]:
tracing = os.getenv("LANGCHAIN_TRACING_V2", "false")
project = os.getenv("LANGCHAIN_PROJECT", "rag-mastery-langgraph")

print(f"حالة تفعيل التتبع (LangSmith Tracing): {tracing}")
print(f"اسم المشروع المسجل: {project}")


حالة تفعيل التتبع (LangSmith Tracing): false
اسم المشروع المسجل: rag-mastery-langgraph


### 4. تجربة تشغيل جراف فحص سريع (Smoke Test)
تعال نبني جرافاً صغيراً جداً مهمته فحص جاهزية النظام، للتأكد من أن محرك LangGraph يعمل بدون أي خطأ.


In [5]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END

class QuickCheckState(TypedDict):
    status: str
    checks: List[str]

def step_one(state: QuickCheckState):
    print("تنفيذ الخطوة الأولى: فحص محرك LangGraph...")
    return {
        "status": "in_progress",
        "checks": state["checks"] + ["LangGraph Engine: OK"]
    }

def step_two(state: QuickCheckState):
    print("تنفيذ الخطوة الثانية: فحص الذاكرة والنواة...")
    return {
        "status": "ready",
        "checks": state["checks"] + ["Kernel & Memory: OK"]
    }

# بناء الجراف
builder = StateGraph(QuickCheckState)
builder.add_node("check_engine", step_one)
builder.add_node("check_kernel", step_two)

builder.add_edge(START, "check_engine")
builder.add_edge("check_engine", "check_kernel")
builder.add_edge("check_kernel", END)

app = builder.compile()

# تشغيل الجراف
result = app.invoke({"status": "starting", "checks": []})
print("\nالنتيجة النهائية لفحص البيئة:")
print(f"الحالة: {result['status']}")
print("سجل الفحوصات:", result["checks"])


تنفيذ الخطوة الأولى: فحص محرك LangGraph...
تنفيذ الخطوة الثانية: فحص الذاكرة والنواة...

النتيجة النهائية لفحص البيئة:
الحالة: ready
سجل الفحوصات: ['LangGraph Engine: OK', 'Kernel & Memory: OK']


### 5. جرب بنفسك! (Hands-on Experiments)
1. **تحدي 1**: أضف خطوة ثالثة في الجراف باسم `check_provider` تفحص هل مفتاح Groq موجود في البيئة وتضيف نتيجة الفحص إلى قائمة `checks`.
2. **تحدي 2**: جرب تغيير مسار الحافة بحيث تتفرع شرطياً: إذا كانت النتيجة ممتازة تذهب مباشرة للنهاية END.
3. **تحدي 3**: افتح موجه الأوامر (Terminal) وجرب الأمر `uv pip list` أو `conda list` وشاهد قائمة الحزم المثبتة في بيئتك المعزولة!
